In [ ]:
import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

In [ ]:
from typing import Dict
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor

import heapq
from tqdm import tqdm
from torch.utils.data import DataLoader

import torchvision.transforms as T
import numpy as np
from PIL import Image

In [ ]:
from loss_distribution.pytorch_script.visual_utils \
	import load_cifar10_data, load_model_state_dict

import torchvision
import torchvision.transforms as transforms

In [ ]:
data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_ds = torchvision.datasets.CIFAR10(root=data_pth, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [ ]:
model_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model = load_model_state_dict('cifar10', 'resnet20', 10, model_path, 'cuda')

In [ ]:
cifar10_classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',     
    'dog', 'frog', 'horse', 'ship', 'truck'
]

In [ ]:
from ntk_result.trials.utils import *

In [ ]:
train_hloss_samples = get_topk_loss_sample(model, train_ds, nn.CrossEntropyLoss(reduction='none'), 16)
test_hloss_samples = get_topk_loss_sample(model, test_ds, nn.CrossEntropyLoss(reduction='none'), 16)

In [ ]:
from torchcam.utils import overlay_mask
from torchcam.methods import GradCAM
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

def generate_gradcam(model, input_tensor, target_layer, device='cuda'):
	input_tensor = input_tensor.to(device)
	input_tensor.requires_grad_(True)
	model = model.to(device)
	model.eval()  

	# 初始化Grad-CAM提取器
	cam_extractor = GradCAM(model, target_layer)

	input_tensor = input_tensor.unsqueeze(0)
	output = model(input_tensor)
	predicted_class = output.squeeze(0).argmax().item()

	# 生成热力图
	activation_map = cam_extractor(predicted_class, output)
	
	# 处理热力图以便可视化
	cam = activation_map[0][0].cpu().detach().numpy()
	cam = np.uint8(255 * cam)
	original_img = inverse_trans_cifar10(input_tensor.squeeze(0).cpu())  # 转回CPU用于可视化
	cam = Image.fromarray(cam).resize(original_img.size, Image.LANCZOS)
	
	result = overlay_mask(original_img, cam, alpha=0.5)

	cam_extractor.remove_hooks()

	return result

In [ ]:
@torch.no_grad()
def eval_sample(
	model, inputs, labels, 
	loss_fn, device='cuda'):

	loss_fn = loss_fn if loss_fn.reduction == 'none' \
					else type(loss_fn)(reduction='none')

	inputs = inputs.to(device)
	model = model.to(device)
	model.eval()

	output = model(inputs)
	loss = loss_fn(output, labels).item()
	pred_idx = torch.argmax(output, dim=1).item()  # 获取预测类别编号

	return loss, pred_idx

In [ ]:
def inverse_trans_cifar10(tensor_img : Tensor) -> Image:
	tensor_img = tensor_img.to('cuda')
	mean = torch.tensor([0.4914, 0.4822, 0.4465], device='cuda').view(3, 1, 1)
	std = torch.tensor([0.2023, 0.1994, 0.2010], device='cuda').view(3, 1, 1)
	tensor_img = tensor_img * std + mean
	
	# 转成numpy数组，shape (H,W,C)
	np_img = tensor_img.mul(255).byte().permute(1, 2, 0).cpu().numpy()
	# 转PIL
	pil_img = Image.fromarray(np_img)
	return pil_img

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(32, 32))  
axes = axes.flatten()
	
for i in range(16):
	sample_idx = train_hloss_samples[i][1]
	cam_res = generate_gradcam(model, train_ds[sample_idx][0], 'layer3.2.conv2')
	axes[i].imshow(cam_res)

# for i in range(16):
# 	sample_idx = train_hloss_samples[i][1]
# 	print(cifar10_classes[pred_cifar10_label_idx(model, train_ds[sample_idx][0])])
for i in range(16):
	sample_idx = train_hloss_samples[i][1]
	true_label = cifar10_classes[
				train_ds[sample_idx][1]]
	pred_label = cifar10_classes[
				eval_sample(model, train_ds[sample_idx][0])]
	print('true: '+true_label+'   pred: '+pred_label)



In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16,16))
axes = axes.flatten()

for i in range(16):
	sample_idx = test_hloss_samples[i][1]
	img = inverse_trans_cifar10(test_ds[sample_idx][0])
	img_show(img, ax=axes[i])

for i in range(16):
	sample_idx = test_hloss_samples[i][1]
	true_label = cifar10_classes[
				test_ds[sample_idx][1]]
	pred_label = cifar10_classes[
				eval_sample(model, test_ds[sample_idx][0])]
	print('true: '+true_label+'   pred: '+pred_label)


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(32, 32))  
axes = axes.flatten()
for i in range(16):
	sample_idx = test_hloss_samples[i][1]
	cam_res = generate_gradcam(model, test_ds[sample_idx][0], 'layer3.2.conv2')
	axes[i].imshow(cam_res)